In [46]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.vectorstores import Chroma
from langchain_classic.docstore.document import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [4]:
vectorstore = Chroma(persist_directory='./vector_db', embedding_function=embeddings)

C:\Users\ohgenenyore\AppData\Local\Temp\ipykernel_7412\4234355290.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory='./vector_db', embedding_function=embeddings)


In [5]:
add_doc= add_doc = Document(metadata={'source': 'Introduction_to_Data_and_Data_Science - Copy.docx'}, page_content='# INTRODUCTION TO RAGS ## Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both')

In [6]:
vectorstore.add_documents([add_doc])

['a1987456-6057-400b-b9ab-6f550c07b3f7']

In [20]:
query = "What softwares are in data science?"

In [8]:
#similarity search
retrieved_docs = vectorstore.similarity_search(query=query, k=5)

for i in retrieved_docs:
    print(i.page_content)

. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources
. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources
. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working 

In [9]:
#MMR search
retrieved_docs2 = vectorstore.max_marginal_relevance_search(query=query, k=5)

for i in retrieved_docs2:
    print(i.page_content)

. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources
. So, let’s move on! ## Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations
. It’s actually a software framework which was designed to address the complexity of b

In [ ]:
retriever = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k':3, 'lambda_mult':0.7})

[Document(metadata={'source': 'Introduction_to_Data_and_Data_Science - Copy.docx'}, page_content='. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science. What about big data? Apart from R and Python, people working in this area are often proficient in other languages like Java or Scala. These two have not been developed specifically for doing statistical analyses, however they turn out to be very useful when combining data from multiple sources'),
 Document(metadata={'source': 'Introduction_to_Data_and_Data_Science - Copy.docx'}, page_content='. So, let’s move on! ## Programming Languages & Software Employed in Data Science - All the Tools You Need Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages 

## GENERATION

In [29]:
template = """
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify what lecture the context is taken from
"""

In [40]:
model = ChatOpenAI(
    model = 'gpt-3.5-turbo',
    temperature = 0.7,
    max_completion_tokens = 100
)

In [32]:
question = "What softwares are in data science?"

In [42]:
prompt = PromptTemplate.from_template(template=template)

In [47]:
chain = {'context': retriever, 'question':RunnablePassthrough()} | prompt | model | StrOutputParser()
chain.invoke(question)

'The softwares commonly used in data science are R, Python, MATLAB, SQL, Java, Scala, Hadoop, Power BI, SaS, Qlik, and Tableau. These tools are essential for working with traditional data, business intelligence, predictive analytics, and big data. This information is sourced from the lecture "Introduction to Data and Data Science".'